# Vector Database Basics

Vector databases help us store, manage, and query the embeddings we created for generative AI, recommenders, and search engines.

Across many of the common use cases, users often find that they need to manage more than just vectors.
To make it easier for practitioners, vector databases should store and manage all of the data they need:
- embedding vectors
- categorical metadata
- numerical metadata
- timeseries metadata
- text / pdf / images / video / point clouds

And support a wide range of query workloads:
- Vector search (may require ANN-index)
- Keyword search (requires full text search index)
- SQL (for filtering)

For this exercise we'll use LanceDB since it's open source and easy to setup

In [1]:
# pip install -U --quiet lancedb pandas pydantic [This has been pre-installed for you]

## Creating tables and adding data

Let's create a LanceDB table called `cats_and_dogs` under the local database directory `~/.lancedb`.
This table should have 4 fields:
- the embedding vector
- a string field indicating the species (either "cat" or "dog")
- the breed
- average weight in pounds

We're going to use pydantic to make this easier. First let's create a pydantic model with those fields

In [2]:
from lancedb.pydantic import vector, LanceModel

class CatsAndDogs(LanceModel):
    vector: vector(2)
    species: str
    breed: str
    weight: float

Now connect to a local db at ~/.lancedb and create an empty LanceDB table called "cats_and_dogs"

In [3]:
import lancedb

db = lancedb.connect('./.lancedb')
table_name='cats_and_dogs'
db.create_table(table_name, schema=CatsAndDogs)

ValueError: Table 'cats_and_dogs' already exists

Let's add some data

First some cats

In [4]:
data = [
    CatsAndDogs(
        vector=[1., 0.],
        species="cat",
        breed="shorthair",
        weight=12.,
    ),
    CatsAndDogs(
        vector=[-1., 0.],
        species="cat",
        breed="himalayan",
        weight=9.5,
    ),
]

Now call the `LanceTable.add` API to insert these two records into the table

In [5]:
cnd_table = db.open_table(table_name)

In [6]:
cnd_table.add(data)

Let's preview the data

In [7]:
cnd_table.head().to_pandas()

,vector,species,breed,weight
0,"[1.0, 0.0]",cat,shorthair,12.0
1,"[-1.0, 0.0]",cat,himalayan,9.5
2,"[0.0, 10.0]",dog,samoyed,47.5
3,"[0.0, -1.0]",dog,corgi,26.0
4,"[1.0, 0.0]",cat,shorthair,12.0


Now let's add some dogs

In [8]:
data = [
    CatsAndDogs(
        vector=[0., 10.],
        species="dog",
        breed="samoyed",
        weight=47.5,
    ),
    CatsAndDogs(
        vector=[0, -1.],
        species="dog",
        breed="corgi",
        weight=26.,
    )
]

In [9]:
cnd_table.add(data)

In [10]:
cnd_table.head().to_pandas()

,vector,species,breed,weight
0,"[1.0, 0.0]",cat,shorthair,12.0
1,"[-1.0, 0.0]",cat,himalayan,9.5
2,"[0.0, 10.0]",dog,samoyed,47.5
3,"[0.0, -1.0]",dog,corgi,26.0
4,"[1.0, 0.0]",cat,shorthair,12.0


## Querying tables

Vector databases allow us to retrieve data for generative AI applications. Let's see how that's done.

Let's say we have a new animal that has embedding [10.5, 10.], what would you expect the most similar animal will be?
Can you use the table we created above to answer the question?

**HINT** you'll need to use the `search` API for LanceTable and `limit` / `to_df` APIs. For examples you can refer to [LanceDB documentation](https://lancedb.github.io/lancedb/basic/#how-to-search-for-approximate-nearest-neighbors).

In [11]:
cnd_table.search([10.5,10]).limit(1).to_pandas()

,vector,species,breed,weight,_distance
0,"[0.0, 10.0]",dog,samoyed,47.5,110.25


Now what if we use cosine distance instead? Would you expect that we get the same answer? Why or why not?

**HINT** you can add a call to `metric` in the call chain

In [12]:
cnd_table.search([10.5,10]).metric('cosine').limit(1).to_pandas()

,vector,species,breed,weight,_distance
0,"[1.0, 0.0]",cat,shorthair,12.0,0.275862


## Filtering tables

In practice, we often need to specify more than just a search vector for good quality retrieval. Oftentimes we need to filter the metadata as well.

Please write code to retrieve two most similar examples to the embedding [10.5, 10.] but only show the results that is a cat.

In [13]:
cnd_table.search([10.5,10]).where('species = "cat"').metric('cosine').limit(3).to_pandas()

,vector,species,breed,weight,_distance
0,"[1.0, 0.0]",cat,shorthair,12.0,0.275862
1,"[1.0, 0.0]",cat,shorthair,12.0,0.275862
2,"[-1.0, 0.0]",cat,himalayan,9.5,1.724138


## Creating ANN indices

For larger tables (e.g., >1M rows), searching through all of the vectors becomes quite slow. Here is where the Approximate Nearest Neighbor (ANN) index comes into play. While there are many different ANN indexing algorithms, they all have the same purpose - to drastically limit the search space as much as possible while losing as little accuracy as possible

For this problem we will create an ANN index on a LanceDB table and see how that impacts performance

### First let's create some data

Given the constraints of the classroom workspace, we'll complete this exercise by creating 100,000 vectors with 16D in a new table. Here the embedding values don't matter, so we simply generate random embeddings as a 2D numpy array. We then use the vec_to_table function to convert that in to an Arrow table, which can then be added to the table.

In [14]:
from lance.vector import vec_to_table
import numpy as np

mat = np.random.randn(100_000, 16)
table_name = "exercise3_ann"
db.drop_table(table_name, ignore_missing=True)
table = db.create_table(table_name, vec_to_table(mat))

/Users/danielfrimer/Projects/Udacity GenAI/udacity_genai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Let's establish a baseline without an index

Before we create the index, let's make sure know what we need to compare against.

We'll generate a random query vector and record it's value in the `query` variable so we can use the same query vector with and without the ANN index.

In [15]:
query = np.random.randn(16)
table.search(query).limit(10).to_df()

/var/folders/6p/nbkwpw5s19q4k695zyrx8g_c0000gn/T/ipykernel_5197/229240492.py:2: UnsupportedWarning: to_df is unsupported as of 0.4.0. Use to_pandas() instead
  table.search(query).limit(10).to_df()


,vector,_distance
0,"[-0.09414049, 0.47841522, -0.75433946, 0.51590...",3.783894
1,"[-0.35663643, 0.10520005, -0.57571644, 1.22795...",4.716362
2,"[-0.56893474, -0.09405723, -0.70170856, 1.6029...",4.896214
3,"[-0.4350346, 0.059806786, -1.2765486, 0.676303...",4.972008
4,"[-0.7587281, -0.8977699, -0.9924041, 1.3307344...",5.054902
5,"[0.07503214, -0.2679368, -1.1394942, 1.1898757...",5.173498
6,"[-1.6470646, -1.0205387, -0.3615188, 0.8998365...",5.682621
7,"[-0.72314787, -0.38986066, -0.2244779, 0.60555...",6.004659
8,"[-0.73933953, -0.19614795, -0.8822928, 0.25488...",6.089478
9,"[-1.4642985, -1.5764948, -0.4413024, 1.0222298...",6.106896


Please write code to compute the average latency of this query

In [16]:
%timeit table.search(query).limit(10).to_df()

<magic-timeit>:1: UnsupportedWarning: to_df is unsupported as of 0.4.0. Use to_pandas() instead


The slowest run took 5.33 times longer than the fastest. This could mean that an intermediate result is being cached.
5.24 ms ± 4.8 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Now let's create an index

There are many possible index types ranging from hash based to tree based to partition based to graph based.
For this task, we'll create an IVFPQ index (partition-based index with product quantization compression) using LanceDB.

Please create an IVFPQ index on the LanceDB table such that each partition is 4000 rows and each PQ subvector is 8D.

**HINT** 
1. Total vectors / number of partitions = number of vectors in each partition
2. Total dimensions / number of subvectors = number of dimensions in each subvector
3. This step can take about 7-10 minutes to process and execute in the classroom workspace.

In [22]:
# table_name = "exercise3_ivfpq"

# mat = np.random.randn(100_000, 16)
# db.drop_table(table_name, ignore_missing=True)
# table = db.create_table(table_name, vec_to_table(mat))
table.create_index(num_partitions=2, num_sub_vectors=2)

In [23]:
%timeit table.search(query).limit(10).to_pandas()

<magic-timeit>:1: UnsupportedWarning: to_df is unsupported as of 0.4.0. Use to_pandas() instead


1.72 ms ± 85.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Now let's search through the data again. Notice how the answers now appear different.
This is because an ANN index is always a tradeoff between latency and accuracy.

In [24]:
table.search(query).limit(10).to_df()

/var/folders/6p/nbkwpw5s19q4k695zyrx8g_c0000gn/T/ipykernel_5197/4174343316.py:1: UnsupportedWarning: to_df is unsupported as of 0.4.0. Use to_pandas() instead
  table.search(query).limit(10).to_df()


,vector,_distance
0,"[0.085922904, -0.7909602, -0.08245702, 1.26824...",4.144537
1,"[-1.4642985, -1.5764948, -0.4413024, 1.0222298...",4.144537
2,"[-0.031172324, 0.6862178, -1.2719188, 2.270290...",4.385981
3,"[0.16839492, -0.38498837, -0.09708845, 1.06098...",4.713158
4,"[-0.5052603, -1.0458591, -0.4074456, 0.4710909...",5.283717
5,"[-0.8693741, -0.28663197, -1.5914031, 0.365995...",5.337337
6,"[-0.35663643, 0.10520005, -0.57571644, 1.22795...",5.343346
7,"[-0.5490025, -0.17556891, 0.07503189, 1.398242...",5.387757
8,"[-0.82717323, 1.0002071, -1.1709205, 1.0024396...",5.387757
9,"[-0.5541169, -1.1003147, -1.5113467, 1.1947556...",5.387757


Now write code to compute the average latency for querying the same table using the ANN index.

**SOLUTION** The index is implementation detail, so it should just be running the same code as above. You should see almost an order of magnitude speed-up. On larger datasets, this performance difference should be even more pronounced.

In [25]:
<fill me in>

SyntaxError: invalid syntax (142683788.py, line 1)

## Deleting rows

Like with other kinds of databases, you should be able to remove rows from the table.
Let's go back to our tables of cats and dogs

In [26]:
table = db["cats_and_dogs"]

In [27]:
len(table)

8

Can you use the `delete` API to remove all of the cats from the table?

**HINT** use a SQL like filter string to specify which rows to delete from the table

In [32]:

table.delete('species = "cat"')

In [33]:
len(table)

4

## What if I messed up?

Errors is a common occurrence in AI. What's hard about errors in vector search is that oftentimes a bad vector doesn't cause a crash but just creates non-sensical answers. So to be able to rollback the state of the database is very important for debugging and reproducibility

So far we've accumulated 4 actions on the table:
1. creation of the table
2. added cats
3. added dogs
4. deleted cats

What if you realized that you should have deleted the dogs instead of the cats?

Here we can see the 4 versions that correspond to the 4 actions we've done

In [34]:
table.list_versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 38, 3, 724766),
  'metadata': {}},
 {'version': 2,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 39, 10, 488203),
  'metadata': {}},
 {'version': 3,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 39, 32, 169232),
  'metadata': {}},
 {'version': 4,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 58, 37, 529159),
  'metadata': {}},
 {'version': 5,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 58, 40, 353217),
  'metadata': {}},
 {'version': 6,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 11, 3, 780218),
  'metadata': {}},
 {'version': 7,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 11, 47, 459874),
  'metadata': {}}]

Please write code to restore the version still containing the whole dataset

In [45]:
table = db["cats_and_dogs"]
table.restore(version=4)

In [46]:
len(table)

6

In [47]:
# restore to version 3

In [48]:
# delete the dogs instead

table.delete('species = "dog"')

In [49]:
table.list_versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 38, 3, 724766),
  'metadata': {}},
 {'version': 2,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 39, 10, 488203),
  'metadata': {}},
 {'version': 3,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 39, 32, 169232),
  'metadata': {}},
 {'version': 4,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 58, 37, 529159),
  'metadata': {}},
 {'version': 5,
  'timestamp': datetime.datetime(2025, 4, 1, 11, 58, 40, 353217),
  'metadata': {}},
 {'version': 6,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 11, 3, 780218),
  'metadata': {}},
 {'version': 7,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 11, 47, 459874),
  'metadata': {}},
 {'version': 8,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 12, 22, 399567),
  'metadata': {}},
 {'version': 9,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 12, 26, 26231),
  'metadata': {}},
 {'version': 10,
  'timestamp': datetime.datetime(2025, 4, 1, 12, 12, 29, 12343),
  'metadata'

In [50]:
table.to_pandas()

,vector,species,breed,weight
0,"[1.0, 0.0]",cat,shorthair,12.0
1,"[-1.0, 0.0]",cat,himalayan,9.5
2,"[1.0, 0.0]",cat,shorthair,12.0
3,"[-1.0, 0.0]",cat,himalayan,9.5


## Dropping a table

You can also choose to drop a table, which also completely removes the data.
Note that this operation is not reversible.

In [51]:
"cats_and_dogs" in db

True

Write code to irrevocably remove the table "cats_and_dogs" from the database

In [52]:
db.drop_table("cats_and_dogs")

How would you verify that the table has indeed been deleted?

In [53]:
table.name in db

False

## Summary

Congrats, in this exercise you've learned the basic operations of vector databases from creating tables, to adding data, and to querying the data. You've learned how to create indices and you saw first hand how it changes the performance and the accuracy. Lastly, you've learned how to debug and rollback when errors happen.